# MandiLense Intelligence — Data Processing Pipeline

## Stage 1: Dataset Configuration and Initial Inspection

In [1]:
import pandas as pd
import numpy as np
import os  #file and folder paths

In [2]:
YEAR = 2025
FILE_PATH = "../data/raw/2025.csv" 

In [3]:
file_size_bytes = os.path.getsize(FILE_PATH)
file_size_gb = file_size_bytes / (1024 ** 3) #to convert it in gb
print(f"File size: {file_size_gb:.2f} GB")

File size: 0.53 GB


Here we are going to do the analysis by chunks cause size of csv file is much biggeer and when we read it ,it get in the RAM 
cause the RAM of my lapotop is only 8 gb so have to do it chunk by chunk.

In [4]:
df_sample=pd.read_csv(
    FILE_PATH,
    nrows=10000
)

In [5]:
df_sample.tail()

,State,District,Market,Commodity,Variety,Grade,Arrival_Date,Min_Price,Max_Price,Modal_Price,Commodity_Code
9995,Haryana,Ambala,Naraingarh,Tomato,Tomato,FAQ,2025-01-01,1000.0,1500.0,1300.0,78
9996,Haryana,Ambala,Naraingarh,Kinnow,Kinnow,Medium,2025-01-01,3000.0,3000.0,3000.0,336
9997,Haryana,Bhiwani,Ch. Dadri,Banana,Other,Large,2025-01-01,1500.0,4000.0,2750.0,19
9998,Haryana,Ambala,Shahzadpur,Carrot,Carrot,FAQ,2025-01-01,1500.0,1500.0,1500.0,153
9999,Haryana,Ambala,Shahzadpur,Cauliflower,Cauliflower,FAQ,2025-01-01,1000.0,1100.0,1000.0,34


In [6]:
df_sample.shape

(10000, 11)

In [7]:
df_sample.columns.tolist()

['State',
 'District',
 'Market',
 'Commodity',
 'Variety',
 'Grade',
 'Arrival_Date',
 'Min_Price',
 'Max_Price',
 'Modal_Price',
 'Commodity_Code']

Stage 2: understanding the data types and quality.

In [8]:
df_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   State           10000 non-null  str    
 1   District        10000 non-null  str    
 2   Market          10000 non-null  str    
 3   Commodity       10000 non-null  str    
 4   Variety         10000 non-null  str    
 5   Grade           10000 non-null  str    
 6   Arrival_Date    10000 non-null  str    
 7   Min_Price       10000 non-null  float64
 8   Max_Price       10000 non-null  float64
 9   Modal_Price     10000 non-null  float64
 10  Commodity_Code  10000 non-null  int64  
dtypes: float64(3), int64(1), str(7)
memory usage: 859.5 KB


In [9]:
df_sample.duplicated().sum()

np.int64(0)

In [10]:
df_sample["Arrival_Date"].head()

0    2025-01-01
1    2025-01-01
2    2025-01-01
3    2025-01-01
4    2025-01-01
Name: Arrival_Date, dtype: str

In [11]:
df_sample["Arrival_Date"].tail()

9995    2025-01-01
9996    2025-01-01
9997    2025-01-01
9998    2025-01-01
9999    2025-01-01
Name: Arrival_Date, dtype: str

as here the datatype of arrival dates is string so we have to change it in the date type 

In [12]:
df_sample["Arrival_Date"]=pd.to_datetime(
    df_sample["Arrival_Date"],
    format="%Y-%m-%d"
    )

In [13]:
df_sample["Arrival_Date"].dtype

dtype('<M8[us]')

Here we are dpomg the chunk of 100000 and then when next chunk comes previous gets earased from ram thats how we uae the ram in efficient way 

In [14]:
chunk_date_ranges = []

for chunk_number, chunk in enumerate(
    pd.read_csv(
        FILE_PATH,
        usecols=["Arrival_Date"],
        chunksize=100_000
    ),
    start=1
):
    chunk["Arrival_Date"] = pd.to_datetime(
        chunk["Arrival_Date"],
        format="%Y-%m-%d"
    )

    chunk_date_ranges.append({
        "chunk": chunk_number,
        "rows": len(chunk),
        "min_date": chunk["Arrival_Date"].min(),
        "max_date": chunk["Arrival_Date"].max()
    })

In [15]:
for r in chunk_date_ranges:
    print(r)

{'chunk': 1, 'rows': 100000, 'min_date': Timestamp('2025-01-01 00:00:00'), 'max_date': Timestamp('2025-01-06 00:00:00')}
{'chunk': 2, 'rows': 100000, 'min_date': Timestamp('2025-01-06 00:00:00'), 'max_date': Timestamp('2025-01-11 00:00:00')}
{'chunk': 3, 'rows': 100000, 'min_date': Timestamp('2025-01-11 00:00:00'), 'max_date': Timestamp('2025-01-17 00:00:00')}
{'chunk': 4, 'rows': 100000, 'min_date': Timestamp('2025-01-17 00:00:00'), 'max_date': Timestamp('2025-01-22 00:00:00')}
{'chunk': 5, 'rows': 100000, 'min_date': Timestamp('2025-01-22 00:00:00'), 'max_date': Timestamp('2025-01-27 00:00:00')}
{'chunk': 6, 'rows': 100000, 'min_date': Timestamp('2025-01-27 00:00:00'), 'max_date': Timestamp('2025-02-01 00:00:00')}
{'chunk': 7, 'rows': 100000, 'min_date': Timestamp('2025-02-01 00:00:00'), 'max_date': Timestamp('2025-02-06 00:00:00')}
{'chunk': 8, 'rows': 100000, 'min_date': Timestamp('2025-02-06 00:00:00'), 'max_date': Timestamp('2025-02-11 00:00:00')}
{'chunk': 9, 'rows': 100000, 'mi

We have done this to check the structure date is good or not means they are in order or not 

We are going to test the each cleaning function here and then going to put into the cleaing.py file 

In [16]:
# in src/cleaning.py
def standardize_columns(df_sample):
    df_sample.columns = (df_sample.columns
                  .str.strip()
                  .str.lower()
                  .str.replace(' ', '_'))
    return df_sample

In [17]:
df_sample=standardize_columns(df_sample)
df_sample.columns

Index(['state', 'district', 'market', 'commodity', 'variety', 'grade',
       'arrival_date', 'min_price', 'max_price', 'modal_price',
       'commodity_code'],
      dtype='str')

In [18]:
def clean_dates(df_sample,date_col="arrival_date"):
    df_sample[date_col]=pd.to_datetime(df_sample["Arrival_Date"],format="%Y-%m-%d",errors='coerce')